# 04 - Modélisation

On entraîne un GradientBoosting avec GridSearchCV et on évalue ses performances.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

load_dotenv("../.env")

BUCKET = os.getenv("S3_BUCKET")
PROC = f"s3://{BUCKET}/processed"

STORAGE = {
    "key": os.getenv("AWS_ACCESS_KEY_ID"),
    "secret": os.getenv("AWS_SECRET_ACCESS_KEY"),
    "client_kwargs": {"region_name": os.getenv("AWS_REGION")}
}

print(f"bucket: {BUCKET}")

## 1. Chargement des données

In [ ]:
df = pd.read_csv(f"{PROC}/dataset_features.csv", storage_options=STORAGE)

FEATURES = [
    "rank_dif",
    "goals_dif", "goals_dif_l5",
    "goals_suf_dif", "goals_suf_dif_l5",
    "goals_per_ranking_dif",
    "dif_rank_agst", "dif_rank_agst_l5",
    "dif_points_rank", "dif_points_rank_l5",
    "is_friendly"
]

X = df[FEATURES]
y = df["target"]

print(f"Dataset : {X.shape[0]} matchs, {X.shape[1]} features")
print(f"Classes : {y.value_counts().to_dict()}")

## 2. Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

print(f"Train : {len(X_train)} matchs")
print(f"Test  : {len(X_test)} matchs")

## 3. Entraînement — GradientBoosting

In [ ]:
gb_params = {
    "learning_rate": [0.01, 0.1, 0.5],
    "min_samples_split": [5, 10],
    "min_samples_leaf": [3, 5],
    "max_depth": [3, 5, 10],
    "max_features": ["sqrt"],
    "n_estimators": [100, 200]
}

gb_cv = GridSearchCV(
    GradientBoostingClassifier(random_state=5),
    gb_params, cv=3, n_jobs=-1, scoring="roc_auc"
)
gb_cv.fit(X_train, y_train)
gb = gb_cv.best_estimator_

auc_train = roc_auc_score(y_train, gb.predict_proba(X_train)[:, 1])
auc_test = roc_auc_score(y_test, gb.predict_proba(X_test)[:, 1])

print(f"GB — train: {auc_train:.3f} | test: {auc_test:.3f}")
print(f"params: {gb_cv.best_params_}")

## 5. Évaluation du modèle

In [ ]:
print(f"GB — train: {auc_train:.3f} | test: {auc_test:.3f} (gap={auc_train - auc_test:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr_test, tpr_test, _ = roc_curve(y_test, best_model.predict_proba(X_test)[:, 1])
fpr_train, tpr_train, _ = roc_curve(y_train, best_model.predict_proba(X_train)[:, 1])

axes[0].plot([0, 1], [0, 1], "k--")
axes[0].plot(fpr_test, tpr_test, label=f"test (AUC={auc_test:.3f})")
axes[0].plot(fpr_train, tpr_train, label=f"train (AUC={auc_train:.3f})")
axes[0].set_title("ROC")
axes[0].legend()

cm = confusion_matrix(y_test, best_model.predict(X_test))
sns.heatmap(cm, annot=True, fmt="d", ax=axes[1])
axes[1].set_title("Matrice de confusion")

plt.tight_layout()
plt.show()

In [ ]:
# Importance des features
importances = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

importances.plot(kind="barh", figsize=(8, 6))
plt.title("Importance des features")
plt.tight_layout()
plt.show()